### Load Files

In [32]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "./knowledge_base",
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

In [33]:
print(documents)

[Document(metadata={'source': 'knowledge_base\\docker_notes.md'}, page_content='# Docker Troubleshooting Notes\n\n## Container keeps restarting\n\n### Problem\n\nA Docker container repeatedly starts and stops.\n\nThis usually means the application inside the container is crashing.\n\n### Common Causes\n\n## 1. Application Error\n\nThe application may have an internal error.\n\nCheck container logs:\n\ndocker logs <container_name>\n\nLook for:\n- stack traces\n- database connection errors\n- missing files\n- configuration errors\n\n## 2. Missing Environment Variables\n\nMany applications require environment variables.\n\nExample:\n\nDATABASE_URL\nAPI_KEY\nPORT\n\nIf these values are missing, the application may fail during startup.\n\nCheck environment variables:\n\ndocker inspect <container_name>\n\n## 3. Incorrect Port Configuration\n\nThe application may listen on a different port than Docker exposes.\n\nExample:\n\nApplication:\n3000\n\nDocker:\n8080\n\nThe ports must be mapped corr

### Chunking

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

In [13]:
print(f"Split {len(documents)} documents into {len(chunks)} chunks")

Split 4 documents into 10 chunks


In [ ]:
print(documents[1])

### Embedding

In [35]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [
    chunk.page_content
    for chunk in chunks
]

embeddings = embedding_model.encode(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8046.29it/s]


In [80]:
print(chunks)

[Document(metadata={'source': 'knowledge_base\\docker_notes.md'}, page_content='# Docker Troubleshooting Notes\n\n## Container keeps restarting\n\n### Problem\n\nA Docker container repeatedly starts and stops.\n\nThis usually means the application inside the container is crashing.\n\n### Common Causes\n\n## 1. Application Error\n\nThe application may have an internal error.\n\nCheck container logs:\n\ndocker logs <container_name>\n\nLook for:\n- stack traces\n- database connection errors\n- missing files\n- configuration errors\n\n## 2. Missing Environment Variables'), Document(metadata={'source': 'knowledge_base\\docker_notes.md'}, page_content='## 2. Missing Environment Variables\n\nMany applications require environment variables.\n\nExample:\n\nDATABASE_URL\nAPI_KEY\nPORT\n\nIf these values are missing, the application may fail during startup.\n\nCheck environment variables:\n\ndocker inspect <container_name>\n\n## 3. Incorrect Port Configuration\n\nThe application may listen on a d

In [ ]:
print(embeddings[0])

### VectorDB

In [96]:
# connect to qdrant
from qdrant_client import QdrantClient

client = QdrantClient(
    host="localhost",
    port=6333
)

print(client.get_collections())

collections=[CollectionDescription(name='opsmind')]


In [ ]:
# define a db collection called opsmind
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name="opsmind",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

In [20]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='opsmind')])

In [117]:
# store ids, chunks & embeddings
from qdrant_client.models import PointStruct

points = []

for i, (chunk, vector) in enumerate(zip(chunks, embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=vector.tolist(),
            payload={
                "text": chunk.page_content,
                "metadata": {
                    "source": chunk.metadata["source"]
                }
            }
        )
    )

client.upsert(
    collection_name="opsmind",
    points=points
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [119]:
client.count(
    collection_name="opsmind"
)

CountResult(count=10)

### Retrieval (without LLM)

In [120]:
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="opsmind",
    embedding=embedding,
    content_payload_key="text",
    metadata_payload_key="metadata",
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.5,
        "k": 3
    }
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4037.51it/s]


In [110]:
query = "How to use docker?"

docs = retriever.invoke(query)

for doc in docs:
    print("SOURCE:", doc.metadata)
    print(doc.page_content)
    print("-"*50)

SOURCE: {'source': 'knowledge_base\\docker_notes.md', '_id': 2, '_collection_name': 'opsmind'}
The ports must be mapped correctly.

Example:

docker run -p 8080:3000 application_name

## Useful Docker Commands

List running containers:

docker ps

View logs:

docker logs <container_name>

View container details:

docker inspect <container_name>

Restart container:

docker restart <container_name>

Remove container:

docker rm <container_name>
--------------------------------------------------
SOURCE: {'source': 'knowledge_base\\docker_notes.md', '_id': 1, '_collection_name': 'opsmind'}
## 2. Missing Environment Variables

Many applications require environment variables.

Example:

DATABASE_URL
API_KEY
PORT

If these values are missing, the application may fail during startup.

Check environment variables:

docker inspect <container_name>

## 3. Incorrect Port Configuration

The application may listen on a different port than Docker exposes.

Example:

Application:
3000

Docker:
8080

T

### Retrieval (with LLM)

In [ ]:
from langchain_ollama.llms import OllamaLLM

model = OllamaLLM(model="llama3.2")

def retrieveByOllamaLLM(query):
    # 1. Retrieve documents
    docs = retriever.invoke(query)

    context = "\n\n".join(
    f"""
    Source: {doc.metadata.get('source')}

    Content:
    {doc.page_content}
    """
        for doc in docs
    )

    # 3. Create prompt
    prompt = f"""
    You are OpsMind, a DevOps assistant.

    Answer using only the provided context.
    Always mention the source file.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # 4. Ask Ollama
    answer = model.invoke(prompt)

    print(answer)

In [122]:
def retrieveByOllamaChat(query):
    from langchain_ollama import ChatOllama

    chat_model = ChatOllama(
        model="llama3.2",
        temperature=0
    )

    from langchain_core.messages import HumanMessage, SystemMessage

    response = chat_model.invoke(
        [
            SystemMessage(
                content="You are a DevOps assistant."
            ),
            HumanMessage(
                content=f"{query}"
            )
        ]
    )

    print(response.content)

In [121]:
retrieveByOllamaLLM("Why is my Kubernetes pod stuck in CrashLoopBackOff?")

Based on the provided context from `knowledge_base\kubernetes_notes.md`, a Kubernetes pod can be stuck in CrashLoopBackOff due to various common causes such as:

1. Application Failure: The application inside the container has an error, which is confirmed by checking logs with `kubectl logs <pod_name>`.
2. Configuration Problem: The application may be missing required ConfigMap values, Secrets, or Environment variables, which can be checked by inspecting pod configuration with `kubectl describe pod <pod_name>`.
3. Resource Limits: The container might not have enough resources, leading to issues such as out-of-memory or CPU limit too low, which can be identified using `kubectl top pods`.

To troubleshoot further, you can use the following commands:

- List all running pods and inspect their details with `kubectl get pods` and `kubectl describe pod <pod_name>`.
- Check service configuration for any errors that might prevent access to your application.
- Use `kubectl logs <pod_name>` to v

In [123]:
retrieveByOllamaChat("Why is my Kubernetes pod stuck in CrashLoopBackOff?")

The dreaded `CrashLoopBackOff`!

When a Kubernetes pod is stuck in `CrashLoopBackOff`, it means that the pod is repeatedly restarting due to one or more of its containers crashing or failing. This can be caused by various issues, including:

1. **Container crashes**: A container within the pod is crashing or exiting with a non-zero exit code.
2. **Resource constraints**: The pod is running out of resources (e.g., memory, CPU) and cannot scale up to meet demand.
3. **Network issues**: Network connectivity problems are preventing the pod from communicating with other pods or services.
4. **Image issues**: The container image is not properly configured or is experiencing issues during startup.
5. **ConfigMap or Secret issues**: Issues with ConfigMaps or Secrets can prevent the pod from accessing necessary configuration data.

To troubleshoot a `CrashLoopBackOff` issue, you can try the following steps:

1. **Check the pod's logs**: Run `kubectl logs <pod-name>` to see the last few lines of

In [65]:
docs = retriever.invoke("How to use docker?")

for doc in docs:
    print("METADATA:")
    print(doc.metadata)

    print("\nCONTENT:")
    print(doc.page_content[:200])

    print("="*50)

METADATA:
{'_id': 2, '_collection_name': 'opsmind'}

CONTENT:

METADATA:
{'_id': 1, '_collection_name': 'opsmind'}

CONTENT:

METADATA:
{'_id': 0, '_collection_name': 'opsmind'}

CONTENT:

